# Shift-aware CRC 3D — prostate multi-site (Kaggle)
Accélérateur : **GPU T4 / P100**. Dépendances : PyTorch (préinstallé) + nibabel.

Inputs : le dataset de code `shift-crc-3d`, le dataset `prostate-multisite`, et (optionnel) la **sortie d'une version précédente de ce notebook** (prétraitement et folds déjà faits sont réutilisés).

**Après tout redémarrage de session, relancer les cellules 1 et 2 avant les autres.**

In [ ]:
!pip install -q --no-deps nibabel
import numpy, torch, nibabel
print('numpy', numpy.__version__, '| torch', torch.__version__, '| nibabel', nibabel.__version__, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU  <-- activer le GPU dans Settings')

In [ ]:
import os, shutil, glob

SITES = ['BIDMC', 'BMC', 'HK', 'I2CVB', 'RUNMC', 'UCL']   # folds (centres cibles) à entraîner dans CETTE session
AUG = 'strong'                 # 'strong' = augmentations de généralisation de domaine (v2)
RUNS = f'runs_{AUG}'           # dossier de sortie des folds
W = '/kaggle/working'

# code : dossier contenant scripts/train_seg.py
hits = glob.glob('/kaggle/input/**/scripts/train_seg.py', recursive=True)
assert hits, "Code introuvable : ajouter le dataset shift-crc-3d en Input"
for h in hits: print('copie du code trouvée :', h, '| v2' if '--aug' in open(h).read() else '| ANCIENNE')
hits = [h for h in hits if '--aug' in open(h).read()]          # garder la version récente du code
assert hits, "Seule une ANCIENNE version du code est en input : mettre à jour le dataset shift-crc-3d"
src = hits[0].rsplit('/scripts/', 1)[0]
# données brutes : dossier contenant BIDMC/*.nii*
raw = glob.glob('/kaggle/input/**/BIDMC/*.nii*', recursive=True)
assert raw, "Données introuvables : aucun fichier BIDMC/*.nii* dans /kaggle/input"
RAW = os.path.dirname(os.path.dirname(raw[0]))
# sortie d'une version précédente (optionnelle) : dossier contenant prostate_npz/
prev = glob.glob('/kaggle/input/**/prostate_npz/metadata.csv', recursive=True)
PREV = os.path.dirname(os.path.dirname(prev[0])) if prev else None
NPZ = f'{W}/prostate_npz'

print('CODE :', src); print('RAW  :', RAW, '->', sorted(os.listdir(RAW))); print('PREV :', PREV)
shutil.copytree(src, f'{W}/code', dirs_exist_ok=True)
os.chdir(f'{W}/code')

## 1. Prétraitement (réutilisé si une sortie précédente est en input)

In [ ]:
if PREV:
    shutil.copytree(f'{PREV}/prostate_npz', NPZ, dirs_exist_ok=True)
    if os.path.isdir(f'{PREV}/{RUNS}'):
        shutil.copytree(f'{PREV}/{RUNS}', f'{W}/{RUNS}', dirs_exist_ok=True)   # folds déjà faits (reprise)
    print('prétraitement réutilisé ; folds déjà présents :', sorted(os.listdir(f'{W}/{RUNS}')) if os.path.isdir(f'{W}/{RUNS}') else [])
else:
    !python scripts/preprocess_prostate.py --raw {RAW} --out {NPZ}

## 2. Entraînement + inférence de chaque fold (~50 min par fold sur T4)
Un fold déjà terminé (summaries.npz présent) est sauté ; un fold interrompu reprend depuis `last.pt`.

In [ ]:
for SITE in SITES:
    RUN = f'{W}/{RUNS}/{SITE}'
    if os.path.exists(f'{RUN}/summaries.npz'):
        print(f'== {SITE} : déjà fait, sauté'); continue
    print(f'\n================ FOLD {SITE} ================')
    !python scripts/train_seg.py --data {NPZ} --target_site {SITE} --out {RUN} --iters 10000 --aug {AUG} --max_hours 1.6
    if not os.path.exists(f'{RUN}/best.pt'):
        raise RuntimeError(f'entraînement de {SITE} échoué : voir le message ci-dessus')
    !python scripts/infer_summaries.py --data {NPZ} --run {RUN} --mc 8

## 3. Analyse conforme sur tous les folds disponibles (CPU, quelques secondes)

In [ ]:
!python scripts/run_conformal.py --runs "{W}/{RUNS}/*" --out {W}/results_{AUG} --reps 200
!python scripts/run_v2.py --runs "{W}/{RUNS}/*" --out {W}/results_{AUG}/v2 --reps 300
# un seul zip avec tous les summaries.npz, nommés par centre (à télécharger depuis l'onglet Output)
import zipfile
with zipfile.ZipFile(f'{W}/summaries_{AUG}.zip', 'w') as z:
    for f in sorted(glob.glob(f'{W}/{RUNS}/*/summaries.npz')):
        z.write(f, f.split(f'{RUNS}/')[1])
print('zip :', f'{W}/summaries_{AUG}.zip')

## 4. Nettoyage avant la fin de la version

In [ ]:
os.chdir(W)
shutil.rmtree(f'{W}/code', ignore_errors=True)
!du -sh {W}/*